# 07 · V4 với phản ứng trước hoán vị quan hệ cú pháp

Bật GPU và Internet. Notebook lấy mã từ nhánh `version4` trên GitHub. Điền `A1_CHECKPOINT` bằng đường dẫn Kaggle Input tới `a1_seed{SEED}.pt` hoặc ZIP checkpoint cùng seed. Notebook chỉ train nhánh R-GCN mới và dùng lại A1.

Nhánh mới dùng cấu hình Abla1 (`struct=0`, `aug=none`) và EMA. Cùng một R-GCN xử lý đồ thị thật và một đồ thị hoán vị nhãn quan hệ trong từng văn bản. Các cạnh ROOT dạng self-loop và số lượng mỗi nhãn được giữ nguyên. Đầu phân loại nhận vector thật cùng hiệu vector thật trừ vector đối chứng. Hai lượt train dùng cùng mặt nạ dropout. Phép hoán vị ở validation/test cố định theo đồ thị, độc lập với batch test.

Checkpoint được chọn bằng tiêu chí validation sạch cộng validation có swap như baseline V4. Hai xác suất human được nhân và phân loại ở ngưỡng 0,5. Kết quả có cả nhánh EMA riêng và EMA × A1 trên HC3 sạch, GPT-3.5 Mixed sạch và HC3 Replace 25%. Đây là thử nghiệm giả thuyết, cần so sánh với V4 Abla1 cùng seed và cùng A1.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, display

WORK = Path('/kaggle/working/prdetect_v4_relation_response')
REPO = WORK / 'PRDetect-Research'
REPO_URL = 'https://github.com/tphan1420/PRDetect-Research.git'
BRANCH = 'version4'
WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
                   check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', REPO_URL, BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'PRDetect {BRANCH}: {commit}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-v3-eval.txt'),
                'torch-geometric'], check=True)
import spacy
if not spacy.util.is_package('en_core_web_sm'):
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings'
print(torch.cuda.get_device_name(0))

In [ ]:
SEED = 2026
V3_EPOCHS = 10
BATCH_SIZE = 1
MICRO_BATCH_SIZE = 1
VAL_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1
# Required: same-seed A1 checkpoint, or a ZIP containing a1_seed{SEED}.pt.
A1_CHECKPOINT = None
CHECKPOINT_DIR = WORK / 'checkpoints'
FEATURE_CACHE = WORK / 'feature_cache'


In [ ]:
if not A1_CHECKPOINT:
    raise ValueError('Set A1_CHECKPOINT to your same-seed A1 file in /kaggle/input.')
a1_path = Path(A1_CHECKPOINT)
if not a1_path.is_file():
    raise FileNotFoundError(a1_path)
if a1_path.suffix.lower() == '.zip':
    name = f'a1_seed{SEED}.pt'
    with ZipFile(a1_path) as reference:
        matches = [n for n in reference.namelist() if Path(n).name == name]
        if len(matches) != 1:
            raise ValueError(f'Expected one {name} in {a1_path}, found {matches}')
        a1_path = WORK / name
        a1_path.write_bytes(reference.read(matches[0]))

cmd = [sys.executable, 'train_v4.py', '--seed', str(SEED),
       '--output-dir', str(CHECKPOINT_DIR), '--feature-cache', str(FEATURE_CACHE),
       '--v3-epochs', str(V3_EPOCHS), '--v3-struct', '0', '--v3-aug', 'none',
       '--relation-response', '--a1-checkpoint', str(a1_path),
       '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE),
       '--val-batch-size', str(VAL_BATCH_SIZE)]
subprocess.run(cmd, cwd=REPO, check=True)
run_dir = CHECKPOINT_DIR / f'seed{SEED}'
training = json.loads((run_dir / 'v4_train.json').read_text(encoding='utf-8'))
assert training['args']['relation_response']
assert training['args']['a1_checkpoint']
archive = run_dir / f'v4_relation_response_hc3_seed{SEED}.zip'
display(FileLink(str(archive)))


In [ ]:
OUT = WORK / f'relation_response_three_tests_seed{SEED}'
subprocess.run([sys.executable, 'v4.py', 'evaluate', '--suite', 'three',
                '--checkpoint-dir', str(CHECKPOINT_DIR), '--seeds', str(SEED), '--batch-size', str(EVAL_BATCH_SIZE),
                '--device', 'cuda', '--output-dir', str(OUT)], cwd=REPO, check=True)
import pandas as pd
results = pd.read_csv(OUT / 'results_by_seed.csv')
expected = {('table2', 'Original'), ('table3', 'Original'), ('table4', 'Replace 25%')}
if len(results) != 3 or set(zip(results['table'], results['condition'])) != expected:
    raise ValueError('Kết quả không khớp ba tập test của notebook 07.')
print('EMA x A1')
display(results)
solo = []
for row in results.to_dict('records'):
    report = json.loads((OUT / f'seed{SEED}' / f"{Path(row['source']).stem}_v3.json").read_text(encoding='utf-8'))
    assert report['relation_response']
    solo.append(dict(table=row['table'], condition=row['condition'], seed=SEED,
                     **{key: report[key] for key in ('accuracy', 'f1', 'auc')}))
solo = pd.DataFrame(solo)
print('EMA')
display(solo)
solo.to_csv(OUT / 'ema_results_by_seed.csv', index=False)
for name in ('v4_train.json', f'v3_baseline_dual_seed{SEED}.train.json'):
    shutil.copyfile(run_dir / name, OUT / name)
(OUT / 'code_commit.txt').write_text(commit + '\n', encoding='utf-8')
archive = shutil.make_archive(str(OUT), 'zip', OUT.parent, OUT.name)
display(FileLink(archive))